In [76]:
import pandas as pd
import json
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms
import torch

In [77]:
# Config
PATH = r"H:\Documents\Ing 3\PFE\data\Data_Projet\mission_herbonaute_2000_seg_black/"

BATCH_SIZE = 10

In [78]:
with open(r"H:\Documents\Ing 3\PFE\data\Data_Projet\data_propre.json", "r") as f:
    data = json.load(f)

In [79]:
df = pd.DataFrame(data)

print(df.head())

          id  epine
0  P03327766      1
1  P04681621      0
2  P03550384      0
3  P01902590      0
4  P02520870      1


In [80]:
print(f"nb ocurence épine {df['epine'].value_counts()}")

nb ocurence épine epine
 1    1169
 0    1108
-1       1
Name: count, dtype: int64


In [81]:
test = df["epine"] == -1
print(df[test])

             id  epine
1647  P04066423     -1


In [82]:
class EpineDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.df.iloc[idx][PATH + "id"]
        label = self.df.iloc[idx][PATH + "epine"]
        image_id = img_path

        img = Image.open(img_path).convert("RGB")
        original_shape = img.size  # (Width, Height)
        im0_shape = torch.tensor(
            [original_shape[1], original_shape[0]], dtype=torch.int
        )  # (H, W)

        if self.transform is not None:
            img = self.transform(img)

        label_tensor = torch.tensor(label, dtype=torch.long)

        metadata_dict = {
            "id": image_id,
            "label": label_tensor,
            "original_shape": im0_shape,
        }

        return img, metadata_dict

In [83]:
IMAGE_SIZE = 224
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

img_size = 640
# Utilisez transforms.Compose pour enchaîner les transformations standard.
transform = transforms.Compose(
    [
        # 1. Convertit l'image PIL en tenseur (C, H, W) et la met entre 0 et 1.
        # C'est la fonction standard de torchvision pour le faire.
        transforms.ToTensor(),
        # 2. Redimensionne l'image.
        # Note: L'argument 'antialias=True' est pris en charge dans les versions récentes
        # de torchvision (>= 0.13), ce qui équivaut souvent à la fonctionnalité de v2.
        transforms.Resize((img_size, img_size), antialias=True),
        # Optionnel: Si vous avez besoin de normaliser l'image APRÈS la conversion et
        # le redimensionnement, vous ajouteriez ceci (bien que YOLO vienne souvent
        # avec sa propre normalisation/préparation):
        # transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
    ]
)

In [84]:
## Dataloader

In [85]:
dataset = EpineDataset(df, transform=transform)

print(len(dataset))

2278


In [86]:
from torch.utils.data import DataLoader

# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size
# train_dataset, val_dataset = torch.utils.data.random_split(
#     dataset, [train_size, val_size]
# )
#
# train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
#
# val_loader = DataLoader(val_dataset,
#                         batch_size=BATCH_SIZE)

loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [93]:
import sys

# On ajoute le répertoire H:\Documents\Ing 3\PFE pour que
# Python trouve le paquet 'YOLOv7_ag'
sys.path.append(r"H:\Documents\Ing 3\PFE\YOLOv7_ag")

In [94]:
from YOLOv7_ag.models.experimental import attempt_load
from YOLOv7_ag.utils.general import non_max_suppression, scale_coords
from YOLOv7_ag.runs.train import *


def run_to_csv(loader, model, tresh, device="cpu"):
    model = attempt_load(model, map_location=device)
    model.eval()
    detections = []
    with torch.no_grad():
        for img, metadata_dict in loader:
            predictions = model(img)[0]
            preditions_list = non_max_suppression(predictions, iou_thres=tresh)
            for i, det in enumerate(preditions_list):
                img_id = metadata_dict["id"][i]
                im0_shape = metadata_dict["original_shape"][i]
                if len(det):
                    det[:, :4] = scale_coords(
                        img.shape[2:], det[:, :4], im0_shape
                    ).round()
                    for *xyxy, conf, cls in det:
                        # Grrr - xyxy est [x1, y1, x2, y2], conf est le score, cls est la classe

                        detections.append(
                            {
                                "image_id": img_id,
                                "x_min": xyxy[0].item(),
                                "y_min": xyxy[1].item(),
                                "x_max": xyxy[2].item(),
                                "y_max": xyxy[3].item(),
                                "confidence": conf.item(),
                                "class_id": cls.item(),
                            }
                        )
    return detections


test = run_to_csv(loader, "best.pt", 0.5)

ModuleNotFoundError: No module named 'project'